# Practical 3 — N-grams & Skip-grams

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To generate n-grams (unigrams, bigrams, trigrams) and skip-grams from the review corpus, and specifically to check whether bigrams can preserve negation context (e.g. "not good") that Practical 2 showed stopword removal puts at risk.

## Theory

A **bag-of-words** representation (single tokens, no order) throws away word order entirely — "not good" and "good not" would look identical, and a single "not" token removed by stopword filtering just vanishes with no trace of what it was negating.

**N-grams** are contiguous sequences of *n* tokens, used to partially recover local word order/context:
- Unigrams (n=1): single tokens — this is what we've been working with so far.
- Bigrams (n=2): adjacent token pairs — e.g. `("not", "good")`.
- Trigrams (n=3): three adjacent tokens.

The key idea for this practical: if you generate bigrams **before** removing stopwords, a pair like `("not", "recommend")` stays intact as a single unit — even if "not" would normally be stripped out on its own. This is one common mitigation for the negation-loss problem from Practical 2.

Trade-off: larger n captures more context but the number of possible n-grams grows very fast, and most specific n-grams appear only once or twice in a small corpus (sparsity) — there's a real cost to this benefit.

**Skip-grams** relax the "adjacent" requirement — they pair tokens that are near each other but allow a gap of up to *k* tokens in between. E.g. with k=1, `"not very good"` produces both the adjacent pair `("not", "very")` and the skip pair `("not", "good")` — which is useful when a modifier sits between a negation and the word it's actually negating.

## Algorithm

1. Reuse cleaned tokens from Practical 1, and stopword-removed tokens from Practical 2.
2. Generate unigrams/bigrams/trigrams for one sample review and inspect them.
3. For the negation reviews identified in Practical 2 (7, 9, 13), generate bigrams **before** and **after** stopword removal, and check whether the negation bigram (e.g. "not recommend") survives in each case.
4. Find the most frequent bigrams across the whole corpus, before and after stopword removal.
5. Generate skip-bigrams (k=1) for one review containing a modifier between a negation and its target word, and compare against plain bigrams.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
from collections import Counter

import preprocessing
import tokenizer
import ngrams

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
df["tokens"] = df["review"].apply(lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r)))

nltk_stops = preprocessing.get_nltk_stopwords()
df["tokens_no_stops"] = df["tokens"].apply(lambda t: preprocessing.remove_stopwords(t, nltk_stops))

print(f"Loaded {len(df)} reviews")


Loaded 15 reviews


### Step 1 — Unigrams, bigrams, trigrams on one review

In [2]:
sample_tokens = df.loc[0, "tokens"]
print("Tokens:  ", sample_tokens)
print("Unigrams:", ngrams.generate_ngrams(sample_tokens, 1))
print("Bigrams: ", ngrams.generate_ngrams(sample_tokens, 2))
print("Trigrams:", ngrams.generate_ngrams(sample_tokens, 3))


Tokens:   ['this', 'movie', 'was', 'absolutely', 'fantastic', 'ive', 'never', 'seen', 'anything', 'like', 'it', 'before']
Unigrams: [('this',), ('movie',), ('was',), ('absolutely',), ('fantastic',), ('ive',), ('never',), ('seen',), ('anything',), ('like',), ('it',), ('before',)]
Bigrams:  [('this', 'movie'), ('movie', 'was'), ('was', 'absolutely'), ('absolutely', 'fantastic'), ('fantastic', 'ive'), ('ive', 'never'), ('never', 'seen'), ('seen', 'anything'), ('anything', 'like'), ('like', 'it'), ('it', 'before')]
Trigrams: [('this', 'movie', 'was'), ('movie', 'was', 'absolutely'), ('was', 'absolutely', 'fantastic'), ('absolutely', 'fantastic', 'ive'), ('fantastic', 'ive', 'never'), ('ive', 'never', 'seen'), ('never', 'seen', 'anything'), ('seen', 'anything', 'like'), ('anything', 'like', 'it'), ('like', 'it', 'before')]


### Step 2 — Does the negation bigram survive stopword removal?

In [3]:
negation_review_ids = [7, 9, 13]  # from Practical 2's Step 3 findings

for review_id in negation_review_ids:
    row = df[df["id"] == review_id].iloc[0]

    bigrams_before = ngrams.generate_ngrams(row["tokens"], 2)
    bigrams_after = ngrams.generate_ngrams(row["tokens_no_stops"], 2)

    negation_bigrams_before = [bg for bg in bigrams_before if "not" in bg or "no" in bg]
    negation_bigrams_after = [bg for bg in bigrams_after if "not" in bg or "no" in bg]

    print(f"Review {review_id}: {row['review']}")
    print(f"  Negation bigrams BEFORE stopword removal: {negation_bigrams_before}")
    print(f"  Negation bigrams AFTER stopword removal:  {negation_bigrams_after}")
    print("-" * 60)


Review 7: Two hours and 15 mins of pure boredom. 2/10 would not recommend to anyone.
  Negation bigrams BEFORE stopword removal: [('would', 'not'), ('not', 'recommend')]
  Negation bigrams AFTER stopword removal:  []
------------------------------------------------------------
Review 9: It's okay... not great, not terrible. Somewhere in the middle I'd say.
  Negation bigrams BEFORE stopword removal: [('okay', 'not'), ('not', 'great'), ('great', 'not'), ('not', 'terrible')]
  Negation bigrams AFTER stopword removal:  []
------------------------------------------------------------
Review 13: Don't get me wrong, it's not BAD, but I expected way more given the hype.
  Negation bigrams BEFORE stopword removal: [('its', 'not'), ('not', 'bad')]
  Negation bigrams AFTER stopword removal:  []
------------------------------------------------------------


### Step 3 — Most frequent bigrams, before vs after stopword removal

In [4]:
top_before = ngrams.top_ngrams(df["tokens"], 2, top_k=10)
top_after = ngrams.top_ngrams(df["tokens_no_stops"], 2, top_k=10)

print("TOP BIGRAMS (before stopword removal):")
for bg, count in top_before:
    print(f"  {bg}: {count}")

print("\nTOP BIGRAMS (after stopword removal):")
for bg, count in top_after:
    print(f"  {bg}: {count}")


TOP BIGRAMS (before stopword removal):
  ('like', 'it'): 3
  ('it', 'was'): 2
  ('of', 'the'): 2
  ('this', 'movie'): 1
  ('movie', 'was'): 1
  ('was', 'absolutely'): 1
  ('absolutely', 'fantastic'): 1
  ('fantastic', 'ive'): 1
  ('ive', 'never'): 1
  ('never', 'seen'): 1

TOP BIGRAMS (after stopword removal):
  ('movie', 'absolutely'): 1
  ('absolutely', 'fantastic'): 1
  ('fantastic', 'ive'): 1
  ('ive', 'never'): 1
  ('never', 'seen'): 1
  ('seen', 'anything'): 1
  ('anything', 'like'): 1
  ('worst', 'film'): 1
  ('film', 'dont'): 1
  ('dont', 'waste'): 1


### Step 4 — Skip-bigrams: catching a negation with a modifier in between

In [5]:
# Review 13 contains "its not bad" -- a modifier-free case.
# Construct a test case where a modifier sits between the negation and its target,
# to see what plain bigrams miss that skip-bigrams catch.
test_tokens = preprocessing.clean_text("It was not very good").split()
print("Test tokens:", test_tokens)

plain_bigrams = ngrams.generate_ngrams(test_tokens, 2)
skip_bigrams = ngrams.generate_skipgrams(test_tokens, n=2, k=1)

print("Plain bigrams:", plain_bigrams)
print("Skip-bigrams (k=1):", skip_bigrams)
print()
print('Does a plain bigram directly pair "not" and "good"?', ("not", "good") in plain_bigrams)
print('Does a skip-bigram directly pair "not" and "good"?', ("not", "good") in skip_bigrams)


Test tokens: ['it', 'was', 'not', 'very', 'good']
Plain bigrams: [('it', 'was'), ('was', 'not'), ('not', 'very'), ('very', 'good')]
Skip-bigrams (k=1): [('it', 'was'), ('it', 'not'), ('was', 'not'), ('was', 'very'), ('not', 'very'), ('not', 'good'), ('very', 'good')]

Does a plain bigram directly pair "not" and "good"? False
Does a skip-bigram directly pair "not" and "good"? True


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- For reviews 7, 9, and 13: did generating bigrams *before* stopword removal actually preserve the negation context that Practical 2 showed was at risk? Give a specific example from your output.
- Looking at the top-10 bigram lists, did removing stopwords make the list more or less informative/specific? Any surprises in what rose to the top?
- Did the skip-bigram result confirm that `("not", "good")` gets captured even with "very" in between, where a plain bigram misses it?
- Given what you found, would you recommend generating bigrams before or after stopword removal for a sentiment analysis task on this kind of data?

### answers:

- Skip-bigrams proved more effective than plain bigrams at capturing relationships separated by an intermediate word: for "it was not very good," a plain bigram could not directly pair "not" and "good" (False), while a skip-bigram (k=1) did (True) — making skip-bigrams better suited to catching modified negation phrases. Comparing the top-10 bigram lists, stopword removal eliminated uninformative pairs like ("it", "was") and ("of", "the"), but also produced an unintended artifact: ("film", "dont") appeared in the "after" list only because removing the stopword "of" from between them created a new adjacency that didn't exist in the original text. Most importantly, all three negation bigrams found intact before stopword removal — ("would", "not")+("not", "recommend") in review 7, ("not", "great")+("not", "terrible") in review 9, and ("not", "bad") in review 13 — were completely eliminated after stopword removal, confirming that generating bigrams before removing stopwords is what actually preserves that context. This shows indiscriminate stopword removal can destroy sentiment-critical information, and that for this dataset, negation words should either be excluded from the stopword list or bigrams should be generated before stopword removal, not after.


---
## Viva Prep — Practice Questions

1. **Why does a bag-of-words / unigram representation struggle with negation?**
   Because it treats each word independently with no positional information — "not" as an isolated token carries no link to the word it was negating, so once separated (e.g. by stopword removal) that relationship is lost entirely.

2. **What's the practical trade-off of increasing n in n-grams?**
   Larger n captures more local context/word order, but the number of distinct n-grams grows rapidly and most of them occur very rarely in any real corpus — leading to a sparse, high-dimensional feature space that's harder to learn from with limited data.

3. **How does a skip-gram differ from a regular n-gram?**
   A regular n-gram requires the tokens to be contiguous; a skip-gram allows a gap of up to k tokens between the paired words, which can catch relationships a plain bigram would miss when a modifier sits in between (e.g. "not very good").

4. **Why generate bigrams before stopword removal rather than after, for a sentiment task?**
   Because if a negation word gets removed by stopword filtering before bigrams are formed, any bigram that would have paired it with the word it negates simply never gets created — the context is destroyed before the bigram step even runs.

5. **What is the sparsity problem in the context of n-grams, and why does it get worse as n increases?**
   As n increases, the number of *possible* n-grams grows combinatorially, but any real corpus only contains a small fraction of them more than once or twice — most n-gram features end up appearing in only one document, giving a model very little to generalize from.
